# Fine-tuning IndoBERT untuk Deteksi Hoaks Bahasa Indonesia

Notebook ini dijalankan di **Google Colab (GPU gratis)**. Tujuannya:

1. Memuat data yang **sudah dibagi by-source** (`scripts/prepare_dataset_id.py`):
   - `train_id_oos.csv` — data latih
   - `test_id_oos.csv` — data uji dari sumber berbeda (**tidak pernah dilatih**)
   - `holdout_id_v2.csv` — hold-out independen (**tidak pernah dilatih/dituning**)
   - **Tidak memakai hold-out v3.** Set v3 hanya untuk laporan akhir dan dievaluasi sekali
     lewat `scripts/evaluate_on_holdout_v3.py` di lokal.
2. Fine-tuning `indobenchmark/indobert-base-p1` pada **revision yang di-pin**
   (`c2cd0b51ddce6580eb35263b39b0a1e5fb0a39e2`) — unduhan deterministik.
3. Reproducibel: `SEED=42`, **early stopping** berdasarkan loss validasi yang juga
   dipisah **by-source** dari data latih.
4. Evaluasi: akurasi + Wilson 95%, ROC-AUC, F1 per kelas, recall hoaks, FPR valid,
   Brier. Semua angka disertai ukuran sampel `n`.
5. Menyimpan model + `metrics.json` + `model_card.md` + `checksums.txt`, lalu
   membungkusnya jadi `indobert.zip` dengan `SHA256SUMS.txt`.

**Penting:** notebook ini TIDAK menentukan `MODEL_RELIABLE`. Flag itu dihitung di
lokal oleh `reliability.py` dari `logs/holdout_v3_result.json` (Butir 4). Metrik
v2 di bawah hanya catatan eksperimen, bukan klaim keandalan.


In [ ]:
# Instal dependensi (sekali per runtime Colab); versi di-pin agar reprodusibel
!pip -q install transformers==4.44.2 datasets evaluate scikit-learn torch sentencepiece protobuf


## Unggah berkas data
Unggah tiga berkas dari folder `data/` lokal ke Colab (tab file → upload):
- `data/train_id_oos.csv`
- `data/test_id_oos.csv`
- `data/holdout_id_v2.csv`

Semua sudah dibagi **by-source** oleh `scripts/prepare_dataset_id.py`.
**Jangan** mengunggah/menyentuh `holdout_id_v3.csv` di notebook ini.
Atau pasang Google Drive dan tuliskan jalurnya di sel berikutnya.


In [ ]:
import os, json, math, glob, random
import hashlib, zipfile
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# --- Konfigurasi reprodusibel ------------------------------------------------
SEED = 42
HF_REVISION = 'c2cd0b51ddce6580eb35263b39b0a1e5fb0a39e2'  # pin commit Hub
MAX_EPOCHS = 6
PATIENCE = 2
random.seed(SEED)
np.random.seed(SEED)

candidates = sorted(glob.glob('/content/*_oos.csv') + glob.glob('/content/*holdout*.csv') + glob.glob('/content/*.csv'))
print('Berkas CSV di /content:')
for c in candidates:
    print(' ', c)

TRAIN_PATH = '/content/train_id_oos.csv'
TEST_PATH = '/content/test_id_oos.csv'
HOLDOUT_PATH = '/content/holdout_id_v2.csv'

# Tidak ada hold-out v3 boleh masuk ke notebook ini.
assert not any('v3' in p for p in (TRAIN_PATH, TEST_PATH, HOLDOUT_PATH)), \
    'Notebook ini dilarang memakai hold-out v3.'
for p in (TRAIN_PATH, TEST_PATH, HOLDOUT_PATH):
    assert os.path.exists(p), f'Berkas tidak ditemukan: {p}. Unggah terlebih dahulu.'

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
holdout = pd.read_csv(HOLDOUT_PATH)
for df in (train, test, holdout):
    df['label'] = df['label'].astype(int)
    assert 'text' in df.columns and 'label' in df.columns

print(f"Train : {len(train)} baris | valid={(train.label==0).sum()} | hoak={(train.label==1).sum()}")
print(f"Test  : {len(test)} baris | valid={(test.label==0).sum()} | hoak={(test.label==1).sum()}")
print(f"Holdout v2: {len(holdout)} baris | valid={(holdout.label==0).sum()} | hoak={(holdout.label==1).sum()}")

# --- Validation split untuk early stopping, juga by-source bila memungkinkan --
if 'source' in train.columns and train['source'].nunique() >= 2:
    srcs = sorted(train['source'].astype(str).unique())
    n_val_src = max(1, int(round(len(srcs) * 0.15)))
    val_srcs = set(srcs[-n_val_src:])
    train_fit = train[~train['source'].astype(str).isin(val_srcs)].reset_index(drop=True)
    val_df = train[train['source'].astype(str).isin(val_srcs)].reset_index(drop=True)
    val_method = f'by-source ({len(val_srcs)} sumber)'
else:
    train_fit, val_df = train_test_split(
        train, test_size=0.15, stratify=train['label'], random_state=SEED)
    val_method = 'random stratified (kolom source tidak tersedia)'
assert len(val_df) > 0 and len(train_fit) > 0, 'Validation split kosong.'
print(f"Early-stopping val: {len(val_df)} baris | {val_method}")
print("Split utama sudah by-source; test & hold-out TIDAK dipakai untuk training/early stopping.")


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, Dataset
from transformers import DataCollatorWithPadding

MODEL_NAME = 'indobenchmark/indobert-base-p1'
MAX_LEN = 256
BATCH_SIZE = 16
LR = 2e-5

# Seed untuk PyTorch (lengkapi seed NumPy/python dari sel sebelumnya).
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
assert device.type == 'cuda', 'GPU tidak tersedia. Ubah runtime ke GPU.'
print('HF revision (pin):', HF_REVISION)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=HF_REVISION)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    revision=HF_REVISION,
    num_labels=2,
    id2label={0: 'valid', 1: 'hoak'},
    label2id={'valid': 0, 'hoak': 1},
).to(device)

class NewsDS(Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(list(texts), truncation=True, padding=True,
                             max_length=MAX_LEN, return_tensors='pt')
        self.labels = torch.tensor(list(labels), dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item['labels'] = self.labels[i]
        return item

collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_loader = DataLoader(NewsDS(train_fit['text'], train_fit['label']),
                           batch_size=BATCH_SIZE, shuffle=True,
                           collate_fn=collator, generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(NewsDS(val_df['text'], val_df['label']),
                         batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)
test_loader = DataLoader(NewsDS(test['text'], test['label']),
                          batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)
holdout_loader = DataLoader(NewsDS(holdout['text'], holdout['label']),
                             batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)
print('DataLoader siap. Train batches:', len(train_loader),
      '| Val batches:', len(val_loader))


In [ ]:
from transformers import get_linear_schedule_with_warmup

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * MAX_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)

@torch.no_grad()
def validation_loss():
    model.eval()
    losses = []
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        losses.append(float(model(**batch).loss.item()))
    model.train()
    return float(np.mean(losses)) if losses else float('inf')

best_val_loss = float('inf')
patience = 0
best_state = None
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(**batch)
        out.loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += float(out.loss.item())
    train_loss = total_loss / len(train_loader)
    val_loss = validation_loss()
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})
    print(f"Epoch {epoch}/{MAX_EPOCHS} - train loss {train_loss:.4f} | val loss {val_loss:.4f}")

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        patience = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience += 1
        print(f"  early stopping: patience {patience}/{PATIENCE}")
        if patience >= PATIENCE:
            print('  Berhenti lebih awal (val loss tidak membaik).')
            break

assert best_state is not None, 'Tidak ada checkpoint terbaik.'
model.load_state_dict(best_state)
model.to(device)
print(f'Fine-tuning selesai. Epoch terbaik dengan val loss={best_val_loss:.4f}')


In [ ]:
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score, recall_score,
    confusion_matrix, brier_score_loss,
)

def evaluate(loader, name):
    model.eval()
    y_true, y_prob, y_pred = [], [], []
    with torch.no_grad():
        for batch in loader:
            labels = batch.pop('labels')
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(**batch).logits
            prob = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            pred = (prob >= 0.5).astype(int)
            y_true.extend(labels.cpu().numpy().tolist())
            y_prob.extend(prob.tolist())
            y_pred.extend(pred.tolist())
    y_true = np.array(y_true); y_prob = np.array(y_prob); y_pred = np.array(y_pred)
    acc = accuracy_score(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = float('nan')
    f1_hoak = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1_valid = f1_score(y_true, y_pred, pos_label=0, zero_division=0)
    rec_hoak = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    valid_mask = (y_true == 0)
    fpr = (y_pred[valid_mask] == 1).mean() if valid_mask.any() else 0.0
    brier = brier_score_loss(y_true, y_prob)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    print(f"\n=== {name} (n={len(y_true)}) ===")
    print(f"Akurasi : {acc:.4f} | ROC-AUC : {auc:.4f}")
    print(f"F1 hoak : {f1_hoak:.4f} | F1 valid : {f1_valid:.4f}")
    print(f"Recall hoak : {rec_hoak:.4f} | FPR (valid->hoak) : {fpr:.4f}")
    print(f"Brier : {brier:.4f} | CM [tn fp; fn tp] = [{tn} {fp}; {fn} {tp}]")
    return {
        'name': name, 'n': int(len(y_true)),
        'n_valid': int((y_true == 0).sum()), 'n_hoax': int((y_true == 1).sum()),
        'accuracy': float(acc), 'roc_auc': float(auc),
        'f1_hoak': float(f1_hoak), 'f1_valid': float(f1_valid),
        'recall_hoak': float(rec_hoak), 'fpr': float(fpr), 'brier': float(brier),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }

res_test = evaluate(test_loader, 'Test out-of-source (sumber berbeda)')
res_holdout = evaluate(holdout_loader, 'Hold-out v2 (manual, independen)')


In [ ]:
# Interval kepercayaan Wilson 95% — setiap angka disertai n.
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (float('nan'), float('nan'))
    p = k / n
    denom = 1 + z*z/n
    centre = (p + z*z/(2*n)) / denom
    half = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / denom
    return (centre-half, centre+half)

def wilson_for_accuracy(res):
    return wilson_ci(round(res['accuracy'] * res['n']), res['n'])

for res in (res_test, res_holdout):
    lo, hi = wilson_for_accuracy(res)
    print(f"{res['name']}: akurasi {res['accuracy']:.2%} "
          f"(n={res['n']}, Wilson 95% [{lo:.2%}, {hi:.2%}])")
    print(f"  recall hoak {res['recall_hoak']:.2%} (n={res['n_hoax']}), "
          f"FPR valid {res['fpr']:.2%} (n={res['n_valid']})")

acc_h = res_holdout['accuracy']
print(f"\nHold-out v2 ≥85%? {'YA' if acc_h >= 0.85 else 'TIDAK'} "
      "(catatan eksperimen; BUKAN penentu MODEL_RELIABLE).")
print("MODEL_RELIABLE dihitung di lokal dari hold-out v3 oleh reliability.py.")


In [ ]:
# Simpan model, metrik, model card, checksum per berkas, dan arsip zip + SHA-256.
OUT_DIR = '/content/indobert_model'
os.makedirs(OUT_DIR, exist_ok=True)
model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

metrics = {
    'model': MODEL_NAME,
    'hf_revision': HF_REVISION,
    'seed': SEED,
    'epochs_max': MAX_EPOCHS,
    'patience': PATIENCE,
    'best_val_loss': float(best_val_loss),
    'history': history,
    'split': {'train_fit': len(train_fit), 'val': len(val_df),
              'val_method': val_method, 'test': len(test),
              'holdout_v2': len(holdout), 'holdout_v3_used': False},
    'test_out_of_source': res_test,
    'holdout_v2': res_holdout,
    'holdout_v2_accuracy_wilson_95': list(wilson_for_accuracy(res_holdout)),
    'note': ('Catatan eksperimen pada hold-out v2. TIDAK menentukan MODEL_RELIABLE; '
             'flag itu dihitung reliability.py dari logs/holdout_v3_result.json.'),
}
with open(os.path.join(OUT_DIR, 'metrics.json'), 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

lo_card, hi_card = wilson_for_accuracy(res_holdout)
card_lines = [
    '# Model Card — IndoBERT (fine-tuned di Colab)',
    '',
    '- Base: `%s` @ revision `%s` (di-pin)' % (MODEL_NAME, HF_REVISION),
    '- Seed: %d | early stopping patience %d (maks %d epoch)' % (SEED, PATIENCE, MAX_EPOCHS),
    '- Split: latih %d, val %d (%s), test %d, hold-out v2 %d'
      % (len(train_fit), len(val_df), val_method, len(test), len(holdout)),
    '- Hold-out v3: TIDAK dipakai di notebook ini',
    '- Test out-of-source: akurasi %.2f%% (n=%d)'
      % (res_test['accuracy'] * 100, res_test['n']),
    '- Hold-out v2: akurasi %.2f%% (n=%d), ROC-AUC %.4f, recall hoak %.2f%%'
      % (res_holdout['accuracy'] * 100, res_holdout['n'],
         res_holdout['roc_auc'], res_holdout['recall_hoax'] * 100),
    '- Wilson 95%% akurasi v2: [%.2f%%, %.2f%%]' % (lo_card * 100, hi_card * 100),
    '- MODEL_RELIABLE: dihitung oleh reliability.py dari hold-out v3, BUKAN dari card ini.',
    '- Keterbatasan: gaya portal/sumber, domain hoak berubah, peluang 0,5 belum dikalibrasi ulang.',
]
with open(os.path.join(OUT_DIR, 'model_card.md'), 'w', encoding='utf-8') as f:
    f.write('\n'.join(card_lines) + '\n')

# checksums.txt: dipakai app.py untuk memuat HANYA jika semua berkas cocok.
skip = {'checksums.txt', 'SHA256SUMS.txt', 'metrics.json', 'model_card.md'}
with open(os.path.join(OUT_DIR, 'checksums.txt'), 'w', encoding='utf-8') as f:
    for name in sorted(os.listdir(OUT_DIR)):
        p = os.path.join(OUT_DIR, name)
        if os.path.isfile(p) and name not in skip:
            h = hashlib.sha256(open(p, 'rb').read()).hexdigest()
            f.write('%s  %s\n' % (h, name))

# Arsip zip + SHA-256 untuk distribusi satu berkas.
ZIP_PATH = '/content/indobert.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in sorted(os.listdir(OUT_DIR)):
        p = os.path.join(OUT_DIR, name)
        if os.path.isfile(p):
            z.write(p, arcname=os.path.join('indobert', name))
zip_hash = hashlib.sha256(open(ZIP_PATH, 'rb').read()).hexdigest()
with open('/content/SHA256SUMS.txt', 'w', encoding='utf-8') as f:
    f.write('%s  indobert.zip\n' % zip_hash)

print('Disimpan:', OUT_DIR)
print('Arsip:', ZIP_PATH)
print('SHA-256 indobert.zip:', zip_hash)
print('Unduh indobert.zip DAN SHA256SUMS.txt, verifikasi hash, lalu ekstrak ke models/indobert/.')


## Langkah selanjutnya (di repo lokal)

1. Unduh `indobert.zip` **dan** `SHA256SUMS.txt` dari Colab.
2. Verifikasi: `sha256sum -c SHA256SUMS.txt` (pastikan `indobert.zip` lolos).
3. Ekstrak isi folder `indobert/` ke `models/indobert/` di repo. Pastikan
   `checksums.txt` ikut ter-extract — `app.py` **hanya memuat IndoBERT bila semua
   checksum di `models/indobert/checksums.txt` cocok**; kalau tidak, ia fallback ke
   model TF-IDF lama (tidak ada model setengah jadi yang dipakai).
4. `MODEL_RELIABLE` **TIDAK** ditentukan oleh `metrics.json`/model card. Flag itu
   dihitung `reliability.py` dari `logs/holdout_v3_result.json` (Butir 4).
5. Hold-out v3 tidak pernah dipakai di notebook; evaluasinya sekali di lokal lewat
   `scripts/evaluate_on_holdout_v3.py`.

Lihat `README_colab.md` untuk panduan lengkap.
